In [ ]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np

In [ ]:
# Leer umbrales sectoriales del Excel (celdas C2 y C3)
umbrales = pd.read_excel("Data/Copia de 4Cuadrantes.xlsx", header=None, usecols=[1, 2], nrows=3)
umbral_roa = umbrales.iloc[1, 1]
umbral_solvencia = umbrales.iloc[2, 1]
print("Umbral ROA:", umbral_roa)
print("Umbral Solvencia:", umbral_solvencia)

# Leer datos de las empresas (desde fila 5)
df = pd.read_excel("Data/Copia de 4Cuadrantes.xlsx", header=4, usecols=[1, 2, 3])
df = df.reset_index(drop=True)

# Calcular la distancia al sector
df["roa"] = df["roa"] - umbral_roa
df["solvencia"] = df["solvencia"] - umbral_solvencia

print()
print(df)


In [ ]:
# 1. Asignar colores vectorizados a todo el dataframe
condiciones = [
    (df["roa"] >= 0) & (df["solvencia"] >= 0),  # Cuadrante 1: PE+, PF+
    (df["roa"] >= 0) & (df["solvencia"] < 0),   # Cuadrante 2: PE+, PF-
    (df["roa"] < 0) & (df["solvencia"] < 0),    # Cuadrante 3: PE-, PF-
    (df["roa"] < 0) & (df["solvencia"] >= 0)    # Cuadrante 4: PE-, PF+
]
colores = ["green", "gold", "red", "orange"]
df["color_cuadrante"] = np.select(condiciones, colores, default="black")

# 2. Crear el gráfico con UNA SOLA TRAZA
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df["solvencia"], 
    y=df["roa"],
    mode="markers+text",
    name="Empresas",
    text=df["nombre"],
    textposition="top center",
    textfont=dict(size=12, color="#2c3e50"),
    marker=dict(size=14, color=df["color_cuadrante"], line=dict(width=1.5, color="#2c3e50")),
    customdata=df[["nombre"]],
    hovertemplate="<b>%{customdata[0]}</b><br>Solvencia: %{x:.2f}<br>ROA: %{y:.2f}<extra></extra>"
))

# Etiquetas de los 4 cuadrantes
cuadrantes = [
    (5, 5, "<b>1</b><br>(PE+, PF+)"),
    (-5, 5, "<b>2</b><br>(PE+, PF-)"),
    (-5, -5, "<b>3</b><br>(PE-, PF-)"),
    (5, -5, "<b>4</b><br>(PE-, PF+)"),
]
for x, y, texto in cuadrantes:
    fig.add_annotation(
        x=x, y=y, text=texto, showarrow=False,
        font=dict(size=14, color="#2c3e50"),
        bordercolor="#2c3e50", borderwidth=1, borderpad=6, bgcolor="white",
    )

# Título eje Y fuera del gráfico
fig.add_annotation(
    x=0, y=1.02, xref="paper", yref="paper",
    text="<b>Posición Económica (+)</b>",
    showarrow=False, font=dict(size=13, color="#2c3e50"), xanchor="center",
)

# Título eje X fuera del gráfico
fig.add_annotation(
    x=1.02, y=0, xref="paper", yref="paper",
    text="<b>Posición Financiera (+)</b>",
    showarrow=False, font=dict(size=13, color="#2c3e50"), xanchor="left", yanchor="middle",
)

# Configurar aspecto
fig.update_layout(
    width=750, height=700,
    plot_bgcolor="white", paper_bgcolor="white",
    xaxis=dict(range=[-15, 15], zeroline=True, zerolinecolor="#2c3e50", zerolinewidth=2, showgrid=False, showticklabels=False),
    yaxis=dict(range=[-15, 15], zeroline=True, zerolinecolor="#2c3e50", zerolinewidth=2, showgrid=False, showticklabels=False, scaleanchor="x", scaleratio=1),
    showlegend=False,
    margin=dict(l=40, r=120, t=50, b=40),
)

fig.show()


Dentro del cluster con empresas desfavorables, hemos elegido hacer este gráfico sobre las empresas del cnae 6820, ya que el cnae que empieza por 68 es uno de los más repetidos en los siguientes clusters y nos interesaba ver como se desempeñaban las empresas ahí. Es el sector de Actividades Inmobiliarias.
cnae 6820:  Alquiler de bienes inmobiliarios por cuenta propia.

La mayoría de empresas con este CNAE se encuentran en riesgo